# 00 - Prepare the shared data (run once, by the maintainer)

Needs the Colab secret `GITHUB_TOKEN` with **Notebook access** switched on for this notebook.

It downloads UIT-HWDB-line into the shared Drive folder, builds `dataset_index.jsonl` and `splits.json` (split by passage), and builds the sentence corpus for role B. Run the cells from top to bottom. If a cell prints an error, copy the whole output to the group.

Outputs on Drive:
- `data/uit_hwdb/<archive>.zip`
- `outputs/dataset_report.txt`, `outputs/dataset_index_all.jsonl` (every labeled line)
- `outputs/dataset_index.jsonl`, `outputs/splits.json`, `outputs/passages.json` (the files the roles use)
- `data/corpus/viwiki_sentences.txt`


## 1. Connect to Drive and GitHub

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

BRANCH = "main"  # use your own branch (a/ocr, b/corrector, c/eval) to test code you have pushed

token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/repo
!git clone -q https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
!git -C /content/repo checkout -q {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "/content/repo")

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"


## 2. Download UIT-HWDB-line into Drive

The file id comes from the dataset README on GitHub. If this cell fails (Google Drive sometimes blocks large downloads), open the link from the README in your browser, make a copy or download it, and put the file into `data/uit_hwdb/` by hand.


In [ ]:
import os
import gdown

LINE_FILE_ID = "1Ul6r2lSP24PlZ6Biabhzg5Q2Wg8Qw6E-"
UIT_DIR = f"{DATA_DIR}/uit_hwdb"
os.makedirs(UIT_DIR, exist_ok=True)

existing = [n for n in os.listdir(UIT_DIR) if not n.startswith(".")]
if existing:
    print("Already in", UIT_DIR, ":", existing)
else:
    try:
        saved = gdown.download(id=LINE_FILE_ID, output=UIT_DIR + "/", quiet=False)
        print("Saved:", saved)
    except Exception as error:
        print("Download failed:", error)
        print("Put the file into", UIT_DIR, "by hand, then run the next cells.")


## 3. Look at the layout (the report is saved to Drive)

In [ ]:
archives = sorted(n for n in os.listdir(UIT_DIR) if n.lower().endswith(".zip"))
print("zip files:", archives)
assert archives, "No .zip file in " + UIT_DIR + ". If the file has another extension, tell the group."
ARCHIVE = f"{UIT_DIR}/{archives[0]}"
!python scripts/inspect_dataset.py "{ARCHIVE}" | tee "{OUT_DIR}/dataset_report.txt"


## 4. Extract on the Colab disk (fast), not on Drive

In [ ]:
import shutil
import zipfile

LOCAL_ROOT = "/content/uit_hwdb"
if not os.path.isdir(LOCAL_ROOT):
    shutil.copy(ARCHIVE, "/content/uit_hwdb.zip")
    with zipfile.ZipFile("/content/uit_hwdb.zip") as archive:
        archive.extractall(LOCAL_ROOT)
print("extracted to", LOCAL_ROOT)


## 5. Build the index and the splits, then check them

The page folder (`train_data/<n>` or `test_data/<n>`) becomes `group_id`. Many pages are different writers copying the same passage, so the splits are made by passage, not by page: the official test stays as it is, pages that share a passage with a test page are left out, and val is made of whole passages.

`build_index.py` writes every labeled line to `dataset_index_all.jsonl`. `make_passage_splits.py` writes the files the roles use (`dataset_index.jsonl`, `splits.json`) and `passages.json` (which pages form a passage, and which pages were left out). The last two commands check the contracts and how much of the val and test text still appears in the training text. If a command stops with an error, copy its full output (and the report from step 3) to the group.


In [ ]:
!python scripts/build_index.py --root {LOCAL_ROOT} --out "{OUT_DIR}/dataset_index_all.jsonl" --group-regex "(?:train_data|test_data)/(\d+)/"
!python scripts/make_passage_splits.py --index "{OUT_DIR}/dataset_index_all.jsonl" --out-index "{OUT_DIR}/dataset_index.jsonl" --out-splits "{OUT_DIR}/splits.json" --report "{OUT_DIR}/passages.json"
!python scripts/validate_dataset_index.py --index "{OUT_DIR}/dataset_index.jsonl" --splits "{OUT_DIR}/splits.json"
!python scripts/check_text_overlap.py --index "{OUT_DIR}/dataset_index.jsonl" --splits "{OUT_DIR}/splits.json"


## 6. Build the sentence corpus for role B (Vietnamese Wikipedia)

The last command checks how many val and test lines already appear word for word in the corpus. If it reports matches, remove those sentences from the corpus before training or scoring the corrector.


In [ ]:
import os

!pip install -q datasets
os.makedirs(f"{DATA_DIR}/corpus", exist_ok=True)
!python scripts/prepare_corpus.py --out "{DATA_DIR}/corpus/viwiki_sentences.txt" --max-sentences 300000
!python scripts/check_text_overlap.py --index "{OUT_DIR}/dataset_index.jsonl" --splits "{OUT_DIR}/splits.json" --corpus "{DATA_DIR}/corpus/viwiki_sentences.txt"


## 7. Summary

In [ ]:
for folder in (UIT_DIR, f"{DATA_DIR}/corpus", OUT_DIR):
    print(folder)
    for name in sorted(os.listdir(folder)):
        size_mb = os.path.getsize(os.path.join(folder, name)) / 1e6
        print("   %-40s %8.1f MB" % (name, size_mb))
